# Sesión 8 · Parte 1 de 3 — Bases de datos y repositorios públicos

Este notebook es autoexplicativo: cada tema incluye su **definición**, la **sintaxis** que se usa,
**para qué sirve** cada elemento, y un **ejemplo ejecutable**.

Esta es la primera de tres partes de la Sesión 8 (Adquisición de datos). Las otras dos son:

- Parte 2 — APIs y autenticación (incluye Kaggle)
- Parte 3 — Web scraping

### Conexión a bases de datos

> **Definición:** Una **base de datos** almacena información de forma organizada, distribuida en una o
> más **tablas**. Cada tabla tiene **registros** (las filas) y **campos** (las columnas). Un programa se
> **conecta** a la base y le envía una **consulta**, que devuelve un **resultado**.

| Concepto | Significado |
|---|---|
| Base de datos | Colección organizada de datos, generalmente con varias tablas |
| Tabla | Conjunto de filas y columnas dentro de la base |
| Registro | Una fila de la tabla |
| Campo | Una columna de la tabla |
| Consulta | La instrucción que solicita datos (escrita en SQL) |
| Resultado | Las filas y columnas que la consulta devuelve |

**Sintaxis:**

```python
import sqlite3
conexion = sqlite3.connect("archivo.db")
df = pd.read_sql("SELECT * FROM tabla", conexion)
```

**Para qué sirve:** es la forma más común de obtener datos dentro de una organización, cuando la
información no se publica como API ni como archivo descargable.

**Fuera de alcance en este curso:** escribir el lenguaje SQL (`SELECT`, `WHERE`, `JOIN`, `GROUP BY`). Aquí
sólo se reconoce el patrón: una conexión + una consulta ya escrita entregan un `DataFrame`, igual que
`read_csv()`.

**Qué hace cada línea del ejemplo:**

- `import sqlite3` — carga el módulo que viene incluido con Python para hablar con bases de datos SQLite. No hay que instalar nada aparte.
- `sqlite3.connect(":memory:")` — abre una conexión. El argumento normalmente es la ruta a un archivo (`"archivo.db"`); `":memory:"` es un valor especial que crea una base de datos temporal en la memoria RAM, sin generar un archivo — útil aquí sólo para practicar.
- `estudiantes.to_sql("estudiantes", conexion, index=False)` — toma un DataFrame que ya existe en Python y lo **transforma en una tabla dentro de la base de datos**, usando la conexión abierta. El primer argumento es el nombre que tendrá la tabla; `index=False` evita que la columna de índice del DataFrame (0, 1, 2, 3...) se guarde como una columna más de la tabla.

En una situación real casi nunca se hace este último paso — la base de datos ya existe de antemano y sólo te conectas a ella. Aquí se crea la tabla `estudiantes` únicamente para tener algo con qué practicar.

In [1]:
import sqlite3
import pandas as pd

# Base de datos local de práctica, creada en memoria
conexion = sqlite3.connect(":memory:")

#Crea un dataframe
estudiantes = pd.DataFrame({
    "nombre": ["Ana", "Luis", "Marco", "Sofia"],
    "calificacion": [92, 78, 85, 96],
})

#Convierte el dataframe en un base datos
estudiantes.to_sql("estudiantes", conexion, index=False)

print("Conexión abierta y tabla creada")

Conexión abierta y tabla creada


### Inspeccionar las tablas disponibles

> **Para qué sirve:** antes de asumir el nombre de una tabla o de sus columnas, conviene preguntarle a la
> propia base de datos qué contiene. SQLite guarda esa información en una tabla especial llamada
> `sqlite_master`.

Estas dos consultas ya están escritas — no es necesario modificarlas ni escribir SQL propio. Sólo se
ejecutan para ver qué hay disponible antes de continuar.

In [8]:
# Qué tablas existen en esta base
consulta_tablas = "SELECT name FROM sqlite_master WHERE type = 'table';"
pd.read_sql(consulta_tablas, conexion)

,name
0,estudiantes


In [9]:
# Qué columnas tiene la tabla "estudiantes", y de qué tipo es cada una
consulta_columnas = "PRAGMA table_info(estudiantes);"
pd.read_sql(consulta_columnas, conexion)

,cid,name,type,notnull,dflt_value,pk
0,0,nombre,TEXT,0,None,0
1,1,calificacion,INTEGER,0,None,0


### Ejecutar una consulta ya escrita

La consulta ya está resuelta — no es necesario escribirla ni modificarla. Lo que hay que reconocer es
qué hace `pd.read_sql()`:

- Recibe dos argumentos: el texto de la consulta SQL, y la conexión abierta a usar.
- Envía esa consulta a la base de datos, recibe el resultado, y lo convierte directamente en un `DataFrame` — sin pasos intermedios.

Es el mismo principio que `read_csv()`: una fuente de datos entra, un `DataFrame` sale.

In [10]:
consulta = "SELECT * FROM estudiantes"
df_desde_bd = pd.read_sql(consulta, conexion)
df_desde_bd

,nombre,calificacion
0,Ana,92
1,Luis,78
2,Marco,85
3,Sofia,96


### Verificar el resultado

> **Para qué sirve:** que una consulta no marque error no significa que el resultado sea el correcto. Antes
> de usar cualquier dato adquirido —de una base de datos, una API o un archivo— conviene revisar:

- cuántas filas y columnas llegaron (`.shape`)
- qué columnas son (`.columns`)
- de qué tipo es cada una (`.dtypes` o `.info()`)
- si hay valores faltantes (`.isna().sum()`)

In [11]:
print("Dimensiones:", df_desde_bd.shape)
print("Columnas:", df_desde_bd.columns.tolist())
print()
df_desde_bd.info()
print()
print("Valores faltantes por columna:")
print(df_desde_bd.isna().sum())

Dimensiones: (4, 2)
Columnas: ['nombre', 'calificacion']

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   nombre        4 non-null      object
 1   calificacion  4 non-null      int64 
dtypes: int64(1), object(1)
memory usage: 196.0+ bytes

Valores faltantes por columna:
nombre          0
calificacion    0
dtype: int64


### Repositorios públicos y descarga directa

> **Definición:** Un **repositorio público de datos** (como Kaggle o UCI Machine Learning Repository) es
> una plataforma donde se publican datasets ya empaquetados, generalmente descargables como `.csv`.

**Cómo se obtiene en la práctica:** se navega al sitio del repositorio, se busca el dataset, y se copia el
enlace de descarga directa del archivo (o se usa la URL "raw" si el dataset vive en un repositorio de
GitHub, como en el ejemplo de abajo).

**Sintaxis:** `read_csv()` no necesita que el archivo esté descargado primero — puede leer directamente
desde una URL.

```python
df = pd.read_csv(url)
```

**Para qué sirve:** este patrón ya se conoce desde Sesión 7; sólo cambia el origen del archivo, de una
ruta local a una dirección web.

El siguiente ejemplo descarga el dataset **Iris**, uno de los datasets más conocidos en ciencia de datos,
publicado originalmente por **UCI Machine Learning Repository**.

In [12]:
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv"
iris = pd.read_csv(url)

print(iris.shape)
iris.head()

(150, 5)


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


> **Definición:** El **diccionario de datos** documenta qué significa cada columna de un dataset. Todo
> dataset descargado —sin importar la fuente— necesita registrarse con su procedencia, fecha de descarga
> y licencia de uso: exactamente el mismo registro de linaje trabajado en el reporte de Data Understanding
> del Proyecto 1. Esta idea se retoma en la Parte 2, al descargar un dataset real desde Kaggle.